In [13]:
import pandas as pd

print("Hello, F1!")

Hello, F1!


In [14]:
drivers = pd.DataFrame({
    "driver": ["Verstappen", "Norris", "Leclerc"],
    "grid": [1, 2, 3],
    "team": ["Red Bull", "McLaren", "Ferrari"]
})
print(drivers["grid"])
drivers

0    1
1    2
2    3
Name: grid, dtype: int64


,driver,grid,team
0,Verstappen,1,Red Bull
1,Norris,2,McLaren
2,Leclerc,3,Ferrari


In [34]:
import requests
import pandas as pd

url = "https://api.jolpi.ca/ergast/f1/2024/circuits/monza/results.json"
response = requests.get(url)
print(response.status_code)

200


In [35]:
data = response.json()
race = data["MRData"]["RaceTable"]["Races"][0]
print(race["raceName"], race["date"])

Italian Grand Prix 2024-09-01


In [42]:
rows = []

for result in race["Results"]:
    rows.append({
        "driver": result["Driver"]["code"],
        "grid": int(result["grid"]),
        "position": int(result["position"]),
        "status": result["status"],
        "team": result["Constructor"]["name"],
        "pos_gained": int(result["grid"]) - int(result["position"])
    })


df = pd.DataFrame(rows)
df = df[["driver", "grid", "position", "status", "pos_gained"]].sort_values("pos_gained")
df


,driver,grid,position,status,pos_gained
16,HUL,10,17,Lapped,-7
6,RUS,3,7,Finished,-4
19,TSU,16,20,Retired,-4
2,NOR,1,3,Finished,-2
18,STR,17,19,Lapped,-2
12,RIC,12,13,Finished,-1
14,GAS,14,15,Lapped,-1
10,ALO,11,11,Finished,0
7,PER,8,8,Finished,0
8,ALB,9,9,Finished,0


In [44]:
def get_race_results(year, round_number):
    url = f"https://api.jolpi.ca/ergast/f1/{year}/{round_number}/results.json"
    response = requests.get(url)
    response.raise_for_status()

    races = response.json()["MRData"]["RaceTable"]["Races"]
    if len(races) == 0:
        return pd.DataFrame()

    race = races[0]
    rows = []
    for result in race["Results"]:
        rows.append({
            "year": int(race["season"]),
            "round": int(race["round"]),
            "circuit": race["Circuit"]["circuitId"],
            "date": race["date"],
            "driver": result["Driver"]["driverId"],
            "team": result["Constructor"]["constructorId"],
            "grid": int(result["grid"]),
            "position": int(result["position"]),
            "status": result["status"],
        })

    return pd.DataFrame(rows)

get_race_results(2024, 16).head()

,year,round,circuit,date,driver,team,grid,position,status
0,2024,16,monza,2024-09-01,leclerc,ferrari,4,1,Finished
1,2024,16,monza,2024-09-01,piastri,mclaren,2,2,Finished
2,2024,16,monza,2024-09-01,norris,mclaren,1,3,Finished
3,2024,16,monza,2024-09-01,sainz,ferrari,5,4,Finished
4,2024,16,monza,2024-09-01,hamilton,mercedes,6,5,Finished


In [45]:
import time

season_tables = []

for round_number in range(1, 31):
    table = get_race_results(2024, round_number)
    if table.empty:
        break
    season_tables.append(table)
    time.sleep(0.5)

season = pd.concat(season_tables, ignore_index=True)
print(len(season_tables), "races,", len(season), "rows")

24 races, 479 rows


In [46]:
from pathlib import Path

raw_folder = Path("../data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

season.to_csv(raw_folder / "results_2024.csv", index=False)

In [49]:
season["driver"].value_counts()

driver
max_verstappen     24
perez              24
leclerc            24
russell            24
hamilton           24
norris             24
piastri            24
alonso             24
hulkenberg         24
stroll             24
zhou               24
tsunoda            24
bottas             24
albon              24
gasly              24
sainz              23
ocon               23
kevin_magnussen    22
ricciardo          18
sargeant           14
colapinto           9
lawson              6
bearman             3
doohan              1
Name: count, dtype: int64